# Day 38 · Same thing in PyTorch: what the library does for you

**First-principles question:** What does PyTorch add on top of your micrograd?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import importlib
import math
import lab.micrograd as micrograd
importlib.reload(micrograd)          # picks up your latest edits to lab/micrograd.py
from lab.micrograd import Value, draw
import time
import torch

## 1 · The neuron in micrograd

In [ ]:
x1 = Value(2.0); x2 = Value(0.0); w1 = Value(-3.0); w2 = Value(1.0); bias = Value(6.8813735870195432)
o = (x1 * w1 + x2 * w2 + bias).tanh()
o.backward()
mine = [x1.grad, x2.grad, w1.grad, w2.grad]
print("micrograd:", o.data, mine)

## 2 · The same neuron in PyTorch
`requires_grad=True` asks PyTorch to remember the graph, like your `_prev`.

In [ ]:
# TODO: make x1, x2, w1, w2, b as torch.tensor([value], dtype=torch.double, requires_grad=True)
#       o = torch.tanh(x1 * w1 + x2 * w2 + b); o.backward()
#       theirs = [x1.grad.item(), x2.grad.item(), w1.grad.item(), w2.grad.item()]
theirs = None
print("pytorch:  ", theirs)

## 3 · Tensors: 1,000 neurons at once

In [ ]:
n = 1000
xs = torch.randn(n, 2, dtype=torch.double)
ws = torch.randn(2, dtype=torch.double, requires_grad=True)
t0 = time.perf_counter()
out = torch.tanh(xs @ ws).sum(); out.backward()
t_torch = time.perf_counter() - t0

import sys
sys.setrecursionlimit(10_000)      # micrograd's backward() walks the graph recursively: 1,000 sums make a deep graph
wv = [Value(float(ws[0])), Value(float(ws[1]))]
t0 = time.perf_counter()
total = sum(((float(r[0]) * wv[0] + float(r[1]) * wv[1]).tanh() for r in xs), Value(0.0)); total.backward()
t_micro = time.perf_counter() - t0
print(f"pytorch {t_torch * 1000:.1f} ms · micrograd {t_micro * 1000:.0f} ms · same gradient? "
      f"{abs(ws.grad[0].item() - wv[0].grad) < 1e-6}")

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert theirs is not None and all(abs(m - t) < 1e-6 for m, t in zip(mine, theirs)), \
    "PyTorch and micrograd gradients should agree"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")